In [8]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from functools import reduce
path = "D:\Khushan_BDA\ABD\SparkSession\Data\Weather Data in India from 1901 to 2017.csv"

In [9]:
spark = SparkSession.builder \
    .appName("Weather Data") \
    .getOrCreate()

In [10]:
df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

In [11]:
df.show()
df.printSchema()

+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|YEAR|  JAN|  FEB|  MAR|  APR|  MAY|  JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|
+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|1901|17.99|19.43|23.49|26.41|28.28| 28.6|27.49|26.98|26.26|25.08|21.73|18.95|
|1902| 19.0|20.39| 24.1|26.54|28.68|28.44|27.29|27.05|25.95|24.37|21.33|18.78|
|1903|18.32|19.79|22.46|26.03|27.93|28.41|28.04|26.63|26.34|24.57|20.96|18.29|
|1904|17.77|19.39|22.95|26.73|27.83|27.85|26.84|26.73|25.84|24.36|21.07|18.84|
|1905| 17.4|17.79|21.78|24.84|28.32|28.69|27.67|27.47|26.29|26.16|22.07|18.71|
|1906| 17.5|19.14|22.21|26.53|29.06|28.02|27.46|26.82|26.23|24.75|21.93|19.55|
|1907|19.27|19.42|22.03|26.98|27.52|27.66|27.28|26.38|26.26|24.72|22.11|18.46|
|1908|18.35|19.73|22.93|27.06|28.07|28.49|27.16|26.41|25.74|24.25|21.06|18.15|
|1909|17.79|19.05| 23.4|25.76|27.97|27.67|26.56|26.43|25.47|24.37|22.01|18.86|
|1910|18.14|19.72| 22.9|25.96|28.36|27.72|26.93|26.6

In [14]:
df = df.withColumn(
    'Avg_Temp',
    round(
        (col('JAN') + col('FEB') + col('MAR') + col('APR') +
         col('MAY') + col('JUN') + col('JUL') + col('AUG') +
         col('SEP') + col('OCT') + col('NOV') + col('DEC')) / 12,
        2
    )
)

df.select('YEAR', 'Avg_Temp').show()

+----+--------+
|YEAR|Avg_Temp|
+----+--------+
|1901|   24.22|
|1902|   24.33|
|1903|   23.98|
|1904|   23.85|
|1905|   23.93|
|1906|    24.1|
|1907|   24.01|
|1908|   23.95|
|1909|   23.78|
|1910|   23.76|
|1911|   23.95|
|1912|   24.11|
|1913|   23.88|
|1914|   24.03|
|1915|   24.29|
|1916|   24.08|
|1917|   23.57|
|1918|   23.83|
|1919|   24.01|
|1920|   23.91|
+----+--------+
only showing top 20 rows


In [15]:
df.select([
    round(avg(col(month)), 2).alias(month)
    for month in months
]).show()

+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+
|  JAN|  FEB|  MAR|  APR|  MAY| JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|
+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+
|18.42|20.14|23.43|26.51|28.39|28.3|27.37|26.94|26.34|24.74|21.77|19.17|
+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+



In [16]:
df = df.withColumn(
    'Min_Temp',
    least(*[col(month) for month in months])
)

df = df.withColumn(
    'Max_Temp',
    greatest(*[col(month) for month in months])
)

df.select('YEAR', 'Min_Temp', 'Max_Temp').show()

+----+--------+--------+
|YEAR|Min_Temp|Max_Temp|
+----+--------+--------+
|1901|   17.99|    28.6|
|1902|   18.78|   28.68|
|1903|   18.29|   28.41|
|1904|   17.77|   27.85|
|1905|    17.4|   28.69|
|1906|    17.5|   29.06|
|1907|   18.46|   27.66|
|1908|   18.15|   28.49|
|1909|   17.79|   27.97|
|1910|   18.05|   28.36|
|1911|   18.52|   28.55|
|1912|   18.44|   28.53|
|1913|    18.2|   27.95|
|1914|   18.73|   28.46|
|1915|   17.93|   29.17|
|1916|   18.17|   28.16|
|1917|   18.16|   27.41|
|1918|   17.25|   28.27|
|1919|   18.58|    28.3|
|1920|   18.27|   27.76|
+----+--------+--------+
only showing top 20 rows


In [17]:
decade_df = df.withColumn(
    'Decade',
    ((col('YEAR') - 1) / 10).cast('int') * 10 + 10
)

In [18]:
decade = decade_df.groupBy('Decade').agg(
    *[
        round(avg(col(month)), 2).alias(month)
        for month in months
    ]
).orderBy('Decade')

decade.show()

+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|Decade|  JAN|  FEB|  MAR|  APR|  MAY|  JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|
+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|  1910|18.15|19.38|22.83|26.28| 28.2|28.16|27.27|26.75|26.04|24.67| 21.5|18.66|
|  1920|18.32|19.62| 22.8|26.01|28.09|28.04|27.37|26.82|26.12| 24.5|21.39|18.52|
|  1930|18.22| 19.9| 23.5|26.48|28.51|28.24|27.28|26.79|26.08|24.35|21.26|18.88|
|  1940| 18.2|19.97|22.95|26.17|28.39|28.16|27.23|26.77|26.26|24.53|21.48|18.97|
|  1950|18.17|19.66|23.24|26.38|28.55|28.35|27.31|26.95|26.21|24.63|21.49|18.94|
|  1960|18.45|20.23|23.43|26.41| 28.3|28.28|27.19|26.83|26.33|24.56|21.53|19.32|
|  1970|18.13|20.14|23.47| 26.3| 28.0|28.22|27.26|26.83|26.24|24.58|21.63|18.98|
|  1980| 18.3|19.87|23.26|26.68|28.14|28.19| 27.2|26.73|26.21|24.75|21.89|19.09|
|  1990| 18.4|20.12|23.32| 26.4|28.13| 28.3|27.31|26.99|26.45|24.68|21.73|19.41|
|  2000|18.69| 20.6|23.68|26

In [19]:
df.select('YEAR', 'Avg_Temp') \
    .orderBy(desc('Avg_Temp')) \
    .limit(1) \
    .show()

+----+--------+
|YEAR|Avg_Temp|
+----+--------+
|2016|   26.45|
+----+--------+



In [20]:
df.select('YEAR', 'Avg_Temp') \
    .orderBy('Avg_Temp') \
    .limit(1) \
    .show()

+----+--------+
|YEAR|Avg_Temp|
+----+--------+
|1917|   23.57|
+----+--------+



In [21]:
df.select(
    'YEAR',
    'Min_Temp'
).orderBy('Min_Temp').limit(1).show()

+----+--------+
|YEAR|Min_Temp|
+----+--------+
|1918|   17.25|
+----+--------+



In [22]:
df.select(
    'YEAR',
    'Max_Temp'
).orderBy(desc('Max_Temp')).limit(1).show()

+----+--------+
|YEAR|Max_Temp|
+----+--------+
|1921|   30.78|
+----+--------+



In [23]:
monthly_rise = df.select([
    round(
        max(col(month)) - min(col(month)),
        2
    ).alias(month)
    for month in months
])

monthly_rise.show()

+----+----+----+----+----+----+----+----+----+----+----+----+
| JAN| FEB| MAR| APR| MAY| JUN| JUL| AUG| SEP| OCT| NOV| DEC|
+----+----+----+----+----+----+----+----+----+----+----+----+
|3.67|5.79|4.83|4.72|3.81|2.55|1.99|1.96|2.64|3.72|3.33|3.91|
+----+----+----+----+----+----+----+----+----+----+----+----+

